# Practica PySpark uso basico

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Configuración mínima (modo local)
spark = (SparkSession.builder
    .appName("PracticaPySpark")
    .master("local[*]")
    .getOrCreate())

empleados = [
    (1, "Ana García",     "Ingeniería",   85000, "CDMX",  8),
    (2, "Carlos López",   "Marketing",    62000, "GDL",   3),
    (3, "María Torres",   "Ingeniería",   92000, "MTY",   12),
    (4, "Juan Martínez",  "Ventas",       58000, "CDMX",  2),
    (5, "Laura Sánchez",  "Ingeniería",   78000, "GDL",   6),
    (6, "Pedro Jiménez",  "Marketing",    71000, "MTY",   9),
    (7, "Sofía Ramírez",  "Ventas",       65000, "CDMX",  4),
    (8, "Andrés Flores",  "Ingeniería",   95000, "GDL",   15),
]

columnas = ["id", "nombre", "departamento", "salario", "ciudad", "antiguedad_años"]
df_emp = spark.createDataFrame(empleados, columnas)


### 1. Muestra el esquema y las primeras 5 filas

In [2]:
df_emp.printSchema()
df_emp.show(5)


root
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: long (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- antiguedad_años: long (nullable = true)

+---+-------------+------------+-------+------+---------------+
| id|       nombre|departamento|salario|ciudad|antiguedad_años|
+---+-------------+------------+-------+------+---------------+
|  1|   Ana García|  Ingeniería|  85000|  CDMX|              8|
|  2| Carlos López|   Marketing|  62000|   GDL|              3|
|  3| María Torres|  Ingeniería|  92000|   MTY|             12|
|  4|Juan Martínez|      Ventas|  58000|  CDMX|              2|
|  5|Laura Sánchez|  Ingeniería|  78000|   GDL|              6|
+---+-------------+------------+-------+------+---------------+
only showing top 5 rows


### 2. Filtra empleados con salario mayor a $70,000

In [3]:
df_emp.filter(F.col("salario") > 70000).show()


+---+-------------+------------+-------+------+---------------+
| id|       nombre|departamento|salario|ciudad|antiguedad_años|
+---+-------------+------------+-------+------+---------------+
|  1|   Ana García|  Ingeniería|  85000|  CDMX|              8|
|  3| María Torres|  Ingeniería|  92000|   MTY|             12|
|  5|Laura Sánchez|  Ingeniería|  78000|   GDL|              6|
|  6|Pedro Jiménez|   Marketing|  71000|   MTY|              9|
|  8|Andrés Flores|  Ingeniería|  95000|   GDL|             15|
+---+-------------+------------+-------+------+---------------+



### 3. Crea una columna bono = 10% del salario si antigüedad > 5 años, 5% si no

In [4]:
df_bono = df_emp.withColumn(
    "bono",
    F.when(F.col("antiguedad_años") > 5, F.col("salario") * 0.10)
     .otherwise(F.col("salario") * 0.05)
)
df_bono.show()


+---+-------------+------------+-------+------+---------------+------+
| id|       nombre|departamento|salario|ciudad|antiguedad_años|  bono|
+---+-------------+------------+-------+------+---------------+------+
|  1|   Ana García|  Ingeniería|  85000|  CDMX|              8|8500.0|
|  2| Carlos López|   Marketing|  62000|   GDL|              3|3100.0|
|  3| María Torres|  Ingeniería|  92000|   MTY|             12|9200.0|
|  4|Juan Martínez|      Ventas|  58000|  CDMX|              2|2900.0|
|  5|Laura Sánchez|  Ingeniería|  78000|   GDL|              6|7800.0|
|  6|Pedro Jiménez|   Marketing|  71000|   MTY|              9|7100.0|
|  7|Sofía Ramírez|      Ventas|  65000|  CDMX|              4|3250.0|
|  8|Andrés Flores|  Ingeniería|  95000|   GDL|             15|9500.0|
+---+-------------+------------+-------+------+---------------+------+



### 4. Muestra el salario promedio por departamento, ordenado de mayor a menor

In [5]:
(df_emp.groupBy("departamento")
    .agg(F.avg("salario").alias("salario_promedio"))
    .orderBy(F.col("salario_promedio").desc())
    .show())


+------------+----------------+
|departamento|salario_promedio|
+------------+----------------+
|  Ingeniería|         87500.0|
|   Marketing|         66500.0|
|      Ventas|         61500.0|
+------------+----------------+



### 5. ¿Cuántos empleados hay en cada ciudad?

In [6]:
df_emp.groupBy("ciudad").count().show()


+------+-----+
|ciudad|count|
+------+-----+
|  CDMX|    3|
|   GDL|    3|
|   MTY|    2|
+------+-----+

